In [321]:
import pandas as pd

from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error 
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import euclidean_distances
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter('ignore')

In [322]:
dataset = pd.read_excel('AmesHousing.xlsx')
dataset

,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,NAmes,1Story
1,2,105000,yes,5,896,882.0,11622,1961,1,2,NAmes,1Story
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,NAmes,1Story
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,NAmes,1Story
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,Gilbert,2Story
...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,142500,yes,6,1003,1003.0,7937,1984,1,3,Mitchel,SLvl
2926,2927,131000,yes,5,902,864.0,8885,1983,1,2,Mitchel,1Story
2927,2928,132000,no,5,970,912.0,10441,1992,1,3,Mitchel,SFoyer
2928,2929,170000,yes,5,1389,1389.0,10010,1974,1,2,Mitchel,1Story


In [323]:
#mogelijke features
SalePrice = dataset['SalePrice']
Garage = pd.get_dummies(dataset.loc[:, 'Garage'], drop_first=True).rename(columns= {'yes' : 'Garage'})
OverallQual = dataset['Overall Qual']
LivArea = dataset['Gr Liv Area']
BsmtSf = dataset['Total Bsmt SF']
LotArea = dataset['Lot Area']
Year = dataset['Year Built']
Bath = dataset['Full Bath']
Bedroom = dataset['Bedroom AbvGr']
Neighbourhood = pd.get_dummies(dataset.loc[:, 'Neighborhood'], drop_first=True)
HouseStyle = pd.get_dummies(dataset.loc[:, 'House Style'], drop_first=True)

In [324]:
standardscalen = True
gebruiken_features = [LivArea, Bedroom, HouseStyle, LotArea]
aantal_clusters = 3
algorithm = 'elkan' #kan kiezen tussen 'elkan'en 'lloyd'
tolerance = 0.1
max_iterations = 2

In [325]:

analyseer = pd.DataFrame()
for x in gebruiken_features:
    analyseer = pd.concat([analyseer, x], axis=1)
columns = analyseer.columns.to_list()
if standardscalen:
    analyseer = StandardScaler().fit_transform(analyseer)
features = analyseer

In [ ]:
KMean = KMeans(n_clusters=aantal_clusters, random_state=42, algorithm=algorithm, tol=tolerance, max_iter=max_iterations)
Kmean = KMean.fit_predict(analyseer)

In [ ]:
kmeans_centre = pd.DataFrame(KMean.cluster_centers_, columns=columns)

In [ ]:
afstanden_tot_centre = pd.DataFrame(euclidean_distances(analyseer, kmeans_centre))

AttributeError: 'numpy.ndarray' object has no attribute 'index'

In [ ]:
for cluster_nummer in range(aantal_clusters):
    afstanden_tot_centre = afstanden_tot_centre.rename(columns = {cluster_nummer: f'Afstand tot cluster_{cluster_nummer}'})
analyseer = pd.concat([analyseer, afstanden_tot_centre], axis = 1) 

In [ ]:
analyseer['Cluster'] = analyseer.iloc[:, -3:].idxmin(axis = 1)
analyseer = analyseer.replace('Afstand tot cluster_0', 'Cluster 0') 
analyseer = analyseer.replace('Afstand tot cluster_1', 'Cluster 1') 
analyseer = analyseer.replace('Afstand tot cluster_2', 'Cluster 2') 
analyseer = analyseer.replace('Afstand tot cluster_3', 'Cluster 3') 

In [ ]:
if len(columns) == (3+aantal_clusters):
    plt.figure(figsize=(8, 6))
    for cluster_id, group in analyseer.groupby('Cluster'):
        plt.scatter( group[analyseer.columns.to_list()[0]], 
        group[analyseer.columns.to_list()[1]], 
        label=f'{cluster_id}',
        alpha=0.7,
        s=40)
else:
    print(analyseer.columns.to_list())

AttributeError: 'numpy.ndarray' object has no attribute 'columns'

In [ ]:

intercluster_matrix = pd.DataFrame(
    euclidean_distances (kmeans_centre, kmeans_centre), 
    index=[f'Cluster {i}' for i in kmeans_centre.index], 
    columns=[f'Cluster {i}' for i in kmeans_centre.index]
)



meanIntercluster = intercluster_matrix.mean().mean()
score = silhouette_score(features, Kmean)
intracluster_mean_per_cluster = {}
for cluster_id in range(aantal_clusters):
    cluster_points = analyseer[analyseer['Cluster'] == f'Cluster {cluster_id}']
    mean_dist = cluster_points[f'Afstand tot cluster_{cluster_id}'].mean()
    intracluster_mean_per_cluster[f'Cluster {cluster_id}'] = mean_dist

mean_intracluster = sum(intracluster_mean_per_cluster.values()) / aantal_clusters

print(f"Intercluster Afstand Mean: {meanIntercluster}")
print(f"Totale Mean Intracluster: {mean_intracluster}")
print(f"Silhouette Score: {score}")
# print("INTRACLUSTERAFSTANDEN VOLGENS DE ELLEBOOGMETHODE")
# inertias = []
# k_values = range(1, 10)
# for k in k_values:
#     kmeans = KMeans(n_clusters=k, random_state=42)
#     kmeans.fit(features)
#     inertias.append(kmeans. inertia_)
# plt.plot(k_values, inertias)
# plt.xlabel("Aantal clusters (k)")
# plt.ylabel("Inertia")
# plt.title("Elleboogcurve")
# plt.show()

Intercluster Afstand Mean: 68646.23911234051
Totale Mean Intracluster: 12195.044166182342
Silhouette Score: 0.7353926936987607
